<a href="https://colab.research.google.com/github/sathundorn/Super-AI-Engineer-Season-6/blob/main/%E0%B9%87Hackaton_3_601402.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
y # 1. ติดตั้งไลบรารี
!pip install -q sentence-transformers faiss-cpu pandas requests

# 2. เชื่อมต่อ Google Drive (จะมีหน้าต่างเด้งให้กดยืนยัน)
from google.colab import drive
drive.mount('/content/drive')

# 3. แตกไฟล์ data.zip (แก้ Path ให้ตรงกับโฟลเดอร์ใน Drive ของคุณ)
!unzip -q /content/drive/MyDrive/dataset/data-2.zip -d /content/fahmai_data/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/fahmai_data/__MACOSX/._data-2? [y]es, [n]o, [A]ll, [N]one, [r]ename: s
error:  invalid response [s]
replace /content/fahmai_data/__MACOSX/._data-2? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/fahmai_data/__MACOSX/._data-2? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [ ]:
!unzip -q /content/drive/MyDrive/dataset/data-2.zip -d /content/fahmai_data/
!find /content/fahmai_data/ -type f -name "*.md" | head -20

replace /content/fahmai_data/__MACOSX/._data-2? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
/content/fahmai_data/__MACOSX/data-2/knowledge_base/policies/._shipping_policy.md
/content/fahmai_data/__MACOSX/data-2/knowledge_base/policies/._cancellation_policy.md
/content/fahmai_data/__MACOSX/data-2/knowledge_base/policies/._warranty_policy.md
/content/fahmai_data/__MACOSX/data-2/knowledge_base/policies/._return_policy.md
/content/fahmai_data/__MACOSX/data-2/knowledge_base/policies/._membership_points_policy.md
/content/fahmai_data/__MACOSX/data-2/knowledge_base/store_info/._buying_guide_smartphones_2569.md
/content/fahmai_data/__MACOSX/data-2/knowledge_base/store_info/._general_faq.md
/content/fahmai_data/__MACOSX/data-2/knowledge_base/store_info/._about_fahmai.md
/content/fahmai_data/__MACOSX/data-2/knowledge_base/products/._DN-LT-015_daonuea_creatorbook_14.md
/content/fahmai_data/__MACOSX/data-2/knowledge_base/products/._SF-SP-014_saifah_phone_senior_plus.md
/content/fahmai_data/__MACOSX/da

In [ ]:
CONFIG = {
    "your_id":       "601402-3",                          # ← รหัสผู้อบรม
    "your_name":     "สธรรดร",                            # ← ชื่อของคุณ
    "kb_dir":        "/content/fahmai_data/data-2/",
    "questions_csv": "/content/fahmai_data/data-2/questions.csv",
    "top_k":         10,
    "model_name":    "Pathumma-ThaiLLM-qwen3-8b-think-3.0.0",    # เปลี่ยนได้
    "use_llm":       True,
}

# Playground Credentials — อัปเดตตรงนี้เมื่อ Session หมดอายุ
CREDS = {
    "url":    "https://playground.thaillm.or.th/api/chat/",
    "csrf":   "h9gPBTCKYV0UcsShOjuFVB60uTTSVV47",        # ← อัปเดตเมื่อหมดอายุ
    "cookie": (
        "csrftoken=h9gPBTCKYV0UcsShOjuFVB60uTTSVV47; "
        "sessionid=zl2dm6rhhf1ikngrggrhli2cokpcrksm; "
        "cf_clearance=B5dj4906ZN5Lj_9sFrlMnJK7L96zgS.7R0vDhN4YHb0-1774790186-1.2.1.1-rQPQHpR0ltLMbaSseZ7Cl8B_fD.Yov4IsG4VddQvVALTChAcW3dwraZQSczPNx6gR.9GjGLGqHXA.s5p1mosqpsb8BHR0TbRUxAlI5ZGP6YIVyu3DzZwo7juf.qmOfZElhWPToFi_sgJURw9ea_Mg5hdZWI3JsYwPXC9cmXS9QLd6.nyRP18i19oNvVYpNxqCdGdnoqPklJlIhR0ux80ovM.j7NMvOZ_9oG4GGV8g0I; "
        "_ga=GA1.1.99528932.1774688726"
    ),
}

print("✅ Config พร้อม")

✅ Config พร้อม


In [ ]:
import re
import json
import time
import numpy as np
import pandas as pd
import requests
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Set, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer


def clean_text(text: str) -> str:
    text = text.replace("\u200b", " ").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_for_match(text: str) -> str:
    text = text.lower()
    text = text.replace("฿", " บาท ").replace("bath", " บาท ")
    text = text.replace("bluetooth", " bluetooth ")
    text = text.replace("wifi", " wifi ").replace("wi-fi", " wifi ")
    text = text.replace("pre-order", " preorder ").replace("trade-in", " tradein ")
    text = re.sub(r"(\d)\s*gb", r"\1gb", text)
    text = re.sub(r"(\d)\s*tb", r"\1tb", text)
    text = re.sub(r"(\d),(\d)", r"\1\2", text)
    text = re.sub(r"[^\w\u0E00-\u0E7F]+", "", text)
    return text


def normalize_loose(text: str) -> str:
    text = text.lower().replace("์", "")
    text = re.sub(r"[^a-z0-9\u0E00-\u0E7F]+", "", text)
    return text


def tokenize(text: str) -> List[str]:
    return re.findall(r"[\w\u0E00-\u0E7F]+", text.lower())


def extract_number_tokens(text: str) -> List[str]:
    return re.findall(r"\d+(?:\.\d+)?", text.replace(",", ""))


def lexical_overlap_score(a: str, b: str) -> float:
    a_norm, b_norm = normalize_for_match(a), normalize_for_match(b)
    if not a_norm or not b_norm:
        return 0.0
    if a_norm in b_norm or b_norm in a_norm:
        return 1.0
    a_t, b_t = set(tokenize(a)), set(tokenize(b))
    return len(a_t & b_t) / max(1, len(a_t | b_t))


def token_recall_score(choice: str, evidence: str) -> float:
    stopwords = {
        "ได้","ไหม","ครับ","ค่ะ","คะ","คือ","และ","หรือ","แบบ","ทุก",
        "ประเภท","สินค้า","รุ่น","การ","ของ","ใน","ที่","มี","เป็น",
        "รองรับ","พร้อม","ฟ้าใหม่","ไม่มี","ข้อมูล","ฐานข้อมูล",
        "สำหรับ","จาก","ด้วย","โดย","แล้ว","ก็","จะ","ให้",
    }
    toks = [t for t in tokenize(choice) if t not in stopwords and len(t) > 1]
    if not toks:
        return 0.0
    ev_toks = set(tokenize(evidence))
    return sum(1 for t in toks if t in ev_toks) / len(toks)


def number_match_score(choice: str, evidence: str) -> float:
    c_nums = extract_number_tokens(choice)
    if not c_nums:
        return 0.0
    e_nums = set(extract_number_tokens(evidence))
    return sum(1 for n in c_nums if n in e_nums) / len(c_nums)


print("✅ Helper functions พร้อม")

✅ Helper functions พร้อม


In [ ]:
def split_markdown_blocks(text: str) -> List[str]:
    blocks, current = [], []
    for raw_line in text.splitlines():
        line = raw_line.rstrip()
        if not line.strip():
            if current:
                blocks.append("\n".join(current).strip())
                current = []
            continue
        if line.startswith("#"):
            if current:
                blocks.append("\n".join(current).strip())
                current = []
            blocks.append(line.strip())
            continue
        if line.strip() == "---":
            if current:
                blocks.append("\n".join(current).strip())
                current = []
            continue
        current.append(line)
    if current:
        blocks.append("\n".join(current).strip())
    return [b for b in blocks if b]


@dataclass
class Chunk:
    chunk_id: str
    source_type: str
    source_name: str
    path: str
    title_path: str
    text: str

    @property
    def combined_text(self) -> str:
        return f"{self.source_name}\n{self.title_path}\n{self.text}".strip()


def parse_markdown_file(path: Path, source_type: str) -> List[Chunk]:
    text = clean_text(path.read_text(encoding="utf-8"))
    blocks = split_markdown_blocks(text)
    title_stack, chunks, buffer = [], [], []
    chunk_index = 0

    def flush():
        nonlocal chunk_index
        if not buffer:
            return
        body = "\n".join(buffer).strip()
        if body:
            tp = " > ".join(title_stack) if title_stack else path.stem
            chunks.append(Chunk(
                chunk_id=f"{path.stem}:{chunk_index}",
                source_type=source_type, source_name=path.stem,
                path=str(path), title_path=tp, text=body,
            ))
            chunk_index += 1
        buffer.clear()

    for block in blocks:
        if block.startswith("#"):
            flush()
            level = len(block) - len(block.lstrip("#"))
            title = block[level:].strip()
            while len(title_stack) >= level:
                title_stack.pop()
            title_stack.append(title)
        else:
            if buffer and len("\n\n".join(buffer + [block])) > 1200:
                flush()
            buffer.append(block)
    flush()
    return chunks


def build_source_aliases(chunks: Sequence[Chunk]) -> Dict[str, Set[str]]:
    aliases: Dict[str, Set[str]] = {}
    for chunk in chunks:
        aliases.setdefault(chunk.source_name, set())
        title_root = chunk.title_path.split(" > ")[0]
        for candidate in {chunk.source_name, title_root,
                          title_root.replace("(", " ").replace(")", " "),
                          chunk.source_name.replace("_", " ")}:
            norm = normalize_loose(candidate)
            if len(norm) >= 4:
                aliases[chunk.source_name].add(norm)
            for part in re.split(r"[\s_/()-]+", candidate):
                pn = normalize_loose(part)
                if len(pn) >= 4:
                    aliases[chunk.source_name].add(pn)
    return aliases


def detect_matching_sources(query: str, aliases: Dict[str, Set[str]]) -> Set[str]:
    qn = normalize_loose(query)
    return {src for src, als in aliases.items() if any(a and a in qn for a in als)}


class KnowledgeBase:
    def __init__(self, chunks: Sequence[Chunk]) -> None:
        self.chunks = list(chunks)
        self.source_aliases = build_source_aliases(self.chunks)
        corpus = [c.combined_text for c in self.chunks]
        # word n-gram (1,3) + char n-gram (2,6) — ดีกว่า baseline
        self.wv = TfidfVectorizer(analyzer="word",    ngram_range=(1,3),
                                   lowercase=False, min_df=1, sublinear_tf=True)
        self.cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,6),
                                   lowercase=False, min_df=1, sublinear_tf=True)
        self.wm = self.wv.fit_transform(corpus)
        self.cm = self.cv.fit_transform(corpus)

    @classmethod
    def from_dir(cls, kb_dir: Path) -> "KnowledgeBase":
        chunks = []
        for stype in ("products", "policies", "store_info"):
            subdir = kb_dir / stype
            if subdir.exists():
                for p in sorted(subdir.glob("*.md")):
                    chunks.extend(parse_markdown_file(p, stype))
        if not chunks:  # fallback: ไม่มี subfolder
            for p in sorted(kb_dir.rglob("*.md")):
                stype = p.parent.name if p.parent != kb_dir else "products"
                chunks.extend(parse_markdown_file(p, stype))
        return cls(chunks)

    def retrieve(self, query: str, top_k: int,
                 source_hint: Optional[str]) -> List[Tuple[Chunk, float]]:
        query = clean_text(query)
        wq = self.wv.transform([query])
        cq = self.cv.transform([query])
        ws = (self.wm @ wq.T).toarray().ravel()
        cs = (self.cm @ cq.T).toarray().ravel()
        scores = 0.55 * ws + 0.45 * cs               # char weight เพิ่มจาก 0.4 → 0.45

        if source_hint:
            scores += np.array([0.10 if c.source_type == source_hint else 0.0
                                 for c in self.chunks])
        matched = detect_matching_sources(query, self.source_aliases)
        if matched:
            scores += np.array([0.25 if c.source_name in matched else 0.0
                                 for c in self.chunks])
        order = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[int(i)], float(scores[int(i)])) for i in order]


# โหลด KB
print("⏳ กำลังโหลด Knowledge Base...")
kb = KnowledgeBase.from_dir(Path(CONFIG["kb_dir"]))
print(f"✅ โหลดสำเร็จ: {len(kb.chunks)} chunks")

⏳ กำลังโหลด Knowledge Base...
✅ โหลดสำเร็จ: 888 chunks


In [ ]:
def detect_source_hint(question: str) -> Optional[str]:
    q = question.lower()
    if any(t in q for t in ["ประกัน","คืนสินค้า","จัดส่ง","ยกเลิก","แต้ม",
                              "wallet","crypto","บัตรเครดิต","ผ่อน","ส่งฟรี",
                              "ค่าส่ง","เคลม","ซ่อม","เปลี่ยน","รับประกัน",
                              "นโยบาย","เงื่อนไข","คืนเงิน","ชำระ","โอน","qr","พร้อมเพย์"]):
        return "policies"
    if any(t in q for t in ["ฟ้าใหม่","สาขา","line","live chat","trade-in","tradein",
                              "ต่างประเทศ","เปิดกี่โมง","เวลาทำการ","ที่อยู่","ติดต่อ",
                              "โทร","facebook","instagram","ออนไลน์","เว็บไซต์","แอป"]):
        return "store_info"
    return "products"


def is_probably_irrelevant(question: str) -> bool:
    q = question.lower()
    # คำที่ไม่เกี่ยวแน่นอน — 1 คำก็พอ
    if any(m in q for m in ["โรงแรม","ปะการัง","กุ้งมังกร","ผลบอล","ดวงชะตา",
                              "หวย","ราศี","ทำนาย","สูตรอาหาร","สปา"]):
        return True
    # คำที่ต้องเจอ >= 2 ตัว
    soft = ["เที่ยว","ภูเก็ต","ร้านอาหาร","ดวง","การเมือง","ท่องเที่ยว",
            "ที่พัก","บินไป","อาหารทะเล","ชายหาด","วัด","พระ"]
    return sum(1 for m in soft if m in q) >= 2


def choose_by_retrieval(question: str, choices: Sequence[str],
                         retrieved: Sequence[Tuple[Chunk, float]]) -> int:
    if is_probably_irrelevant(question):
        return 10

    evidence_text = "\n".join(c.text for c, _ in retrieved[:5])
    best_score    = retrieved[0][1] if retrieved else 0.0
    scores        = []

    for idx, choice in enumerate(choices, start=1):
        if idx >= 9:
            scores.append(-1.0)
            continue
        num  = number_match_score(choice, evidence_text)
        score = (
            0.40 * lexical_overlap_score(choice, evidence_text)
            + 0.25 * token_recall_score(choice, evidence_text)
            + 0.18 * lexical_overlap_score(choice, question)
            + 0.12 * num
            + 0.05 * best_score
        )
        if num == 1.0 and extract_number_tokens(choice):
            score += 0.18
        # negation patterns
        for kw in [("ไม่มีบริการ","ไม่มีบริการ"), ("ไม่รับ","ไม่รับ"),
                   ("ไม่รองรับ","ไม่รองรับ"), ("pre-order","pre-order"),
                   ("preorder","preorder"), ("ไม่สามารถ","ไม่สามารถ"),
                   ("ไม่จัดส่ง","ไม่จัดส่ง"), ("ไม่คืน","ไม่คืน")]:
            if kw[0] in evidence_text.lower() and kw[1] in choice.lower():
                score += 0.22
        scores.append(score)

    best_choice       = int(np.argmax(scores)) + 1
    best_choice_score = scores[best_choice - 1]

    if best_score < 0.04 and best_choice_score < 0.15:
        return 9
    if best_choice_score < 0.06:
        return 9
    return best_choice


def build_query(question: str, choices: Sequence[str]) -> str:
    irrelevant = {"ไม่มีข้อมูลนี้ในฐานข้อมูล", "คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า"}
    return clean_text(question + "\n" + "\n".join(c for c in choices if c not in irrelevant))


print("✅ Retrieval + Heuristic functions พร้อม")

✅ Retrieval + Heuristic functions พร้อม


In [ ]:
def build_system_prompt() -> str:
    return (
        "คุณเป็นผู้ช่วยตอบ multiple-choice ของร้านฟ้าใหม่เท่านั้น "
        "เลือกตัวเลือกที่ตรงกับหลักฐานมากที่สุดจากข้อ 1-10 เพียงข้อเดียว\n"
        "กติกา:\n"
        "1) ยึดข้อความในหลักฐานแบบตรงตัว — ราคา รุ่น สี จำนวน ปี เดือน หน่วย "
        "มาตรฐาน IP Bluetooth ความจุ พอร์ต เงื่อนไขนโยบาย\n"
        "2) ถ้าหลักฐานบอกปฏิเสธ (ไม่มีบริการ ไม่รับ ไม่รองรับ ฯลฯ) "
        "ให้เลือกตัวเลือกเชิงปฏิเสธที่ตรงที่สุด\n"
        "3) ผลเบื้องต้นเป็นแค่ hint ถ้าหลักฐานชี้ตัวเลือกอื่นชัดกว่า ให้เลือกตัวเลือกนั้น\n"
        "4) เลือกข้อ 9 เมื่อไม่พบข้อมูลในหลักฐานจริงๆ\n"
        "5) เลือกข้อ 10 เมื่อคำถามไม่เกี่ยวกับร้านฟ้าใหม่เลย\n"
        "6) ห้ามเดาเกินหลักฐาน\n"
        "ตอบเป็น JSON บรรทัดเดียว: {\"answer\": <1-10>, \"reason\": \"สั้นๆ\"}"
    )


def build_user_prompt(question: str, choices: Sequence[str],
                       retrieved: Sequence[Tuple[Chunk, float]],
                       heuristic_answer: int) -> str:
    evidence = []
    for i, (chunk, score) in enumerate(retrieved, start=1):
        evidence.append(
            f"[หลักฐาน {i}] source={chunk.source_name}; "
            f"title={chunk.title_path}; score={score:.4f}\n{chunk.text[:900]}"
        )
    choices_block = "\n".join(f"{i}. {c}" for i, c in enumerate(choices, start=1))
    return (
        f"คำถาม:\n{question}\n\n"
        f"ตัวเลือก:\n{choices_block}\n\n"
        f"ผลเบื้องต้น: ข้อ {heuristic_answer} "
        f"(ตรวจสอบด้วยตัวเองและแก้ถ้าหลักฐานบ่งชี้ต่างออกไป)\n\n"
        f"หลักฐาน:\n" + "\n\n".join(evidence)
    )


def call_playground(full_prompt: str) -> str:
    headers = {
        "Accept": "*/*",
        "Content-Type": "application/json",
        "Origin": "https://playground.thaillm.or.th",
        "Referer": "https://playground.thaillm.or.th/chat/",
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.6 Safari/605.1.15"
        ),
        "X-CSRFToken": CREDS["csrf"],
        "Cookie":      CREDS["cookie"],
    }
    data = {
        "model":    CONFIG["model_name"],
        "messages": [{"role": "user", "content": full_prompt}],
        "temperature": 0.0,
    }
    resp = requests.post(CREDS["url"], headers=headers, json=data, timeout=45)
    resp.raise_for_status()
    result = resp.json()
    if "choices" in result:
        return result["choices"][0]["message"]["content"].strip()
    elif "message" in result:
        return result["message"].strip()
    elif "content" in result:
        return result["content"].strip()
    raise RuntimeError(f"Unknown response format: {result}")


def ask_llm(question: str, choices: Sequence[str],
             retrieved: Sequence[Tuple[Chunk, float]],
             heuristic_answer: int, max_retries: int = 3) -> Optional[str]:
    system  = build_system_prompt()
    user    = build_user_prompt(question, choices, retrieved, heuristic_answer)
    full    = f"{system}\n\n{user}"
    for attempt in range(1, max_retries + 1):
        try:
            return call_playground(full)
        except Exception as e:
            print(f"  ⚠️  Retry {attempt}/{max_retries} ({e})")
            if attempt < max_retries:
                time.sleep(3 * attempt)
    return None


def parse_llm_answer(text: str) -> Optional[int]:
    try:
        cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
        cleaned = re.sub(r"```.*?```", "", cleaned, flags=re.DOTALL)
        m = re.search(r"\{.*?\}", cleaned, re.DOTALL)
        if m:
            data   = json.loads(m.group())
            answer = int(data["answer"])
            if 1 <= answer <= 10:
                return answer
    except Exception:
        pass
    m = re.search(r"\b([1-9]|10)\b", text)
    if m:
        a = int(m.group(1))
        if 1 <= a <= 10:
            return a
    return None


print("✅ LLM + Prompt builder พร้อม")

✅ LLM + Prompt builder พร้อม


In [ ]:
from collections import Counter

questions_df = pd.read_csv(CONFIG["questions_csv"])
results      = []
error_count  = 0

print(f"🚀 เริ่มทำข้อสอบ {len(questions_df)} ข้อ...\n")

for _, row in questions_df.iterrows():
    q_id     = int(row["id"])
    question = str(row["question"]).strip()
    choices  = [str(row[f"choice_{i}"]).strip() for i in range(1, 11)]

    # Retrieve
    retrieved        = kb.retrieve(build_query(question, choices),
                                   top_k=CONFIG["top_k"],
                                   source_hint=detect_source_hint(question))
    heuristic_answer = choose_by_retrieval(question, choices, retrieved)

    # LLM verify
    final_answer = heuristic_answer
    if CONFIG["use_llm"]:
        raw = ask_llm(question, choices, retrieved, heuristic_answer)
        if raw is None:
            error_count += 1
        else:
            parsed = parse_llm_answer(raw)
            if parsed is not None:
                final_answer = parsed

    results.append({"id": q_id, "answer": final_answer})
    print(f"ข้อที่ {q_id:>3} | heuristic={heuristic_answer} | final={final_answer}")

print(f"\n✅ เสร็จสิ้น! LLM error (fallback): {error_count} ข้อ")

🚀 เริ่มทำข้อสอบ 100 ข้อ...

ข้อที่   1 | heuristic=3 | final=5
ข้อที่   2 | heuristic=7 | final=7
ข้อที่   3 | heuristic=2 | final=2
ข้อที่   4 | heuristic=6 | final=6
ข้อที่   5 | heuristic=6 | final=1
ข้อที่   6 | heuristic=5 | final=8
ข้อที่   7 | heuristic=1 | final=1
ข้อที่   8 | heuristic=4 | final=4
ข้อที่   9 | heuristic=10 | final=4
ข้อที่  10 | heuristic=2 | final=10
ข้อที่  11 | heuristic=6 | final=4
ข้อที่  12 | heuristic=1 | final=1
ข้อที่  13 | heuristic=1 | final=6
ข้อที่  14 | heuristic=2 | final=5
ข้อที่  15 | heuristic=7 | final=5
ข้อที่  16 | heuristic=3 | final=1
ข้อที่  17 | heuristic=8 | final=1
ข้อที่  18 | heuristic=5 | final=1
ข้อที่  19 | heuristic=2 | final=1
ข้อที่  20 | heuristic=8 | final=1
ข้อที่  21 | heuristic=5 | final=3
  ⚠️  Retry 1/3 (503 Server Error: Service Unavailable for url: https://playground.thaillm.or.th/api/chat/)
ข้อที่  22 | heuristic=8 | final=2
ข้อที่  23 | heuristic=6 | final=3
ข้อที่  24 | heuristic=1 | final=6
ข้อที่  25 | heuristic

In [ ]:
submission_df = pd.DataFrame(results)

file_name = f"submission_{CONFIG['your_id']}-{CONFIG['your_name']}.csv"
submission_df.to_csv(file_name, index=False)

print(f"📄 บันทึกไฟล์: {file_name}")
print(f"📊 ทำสำเร็จ: {len(results) - error_count}/{len(results)} ข้อ")

dist = Counter(r["answer"] for r in results)
print(f"\n📈 การกระจายคำตอบ:")
for ans in sorted(dist):
    bar = "█" * dist[ans]
    print(f"  {ans:>2}: {bar} ({dist[ans]})")

📄 บันทึกไฟล์: submission_601402-3-สธรรดร.csv
📊 ทำสำเร็จ: 100/100 ข้อ

📈 การกระจายคำตอบ:
   1: ██████████████████████████████ (30)
   2: ███████████ (11)
   3: ██████ (6)
   4: ███████ (7)
   5: █████████████ (13)
   6: █████████ (9)
   7: ██████ (6)
   8: █████ (5)
   9: ███████████ (11)
  10: ██ (2)
